In [17]:
import numpy as np
import pandas as pd
from decimal import Decimal, ROUND_DOWN
np.random.seed(0)

train = pd.read_csv('dataset_train.csv')
test = pd.read_csv('dataset_eval.csv')

SUBTASK1

In [22]:
def monthit(sentence):
    return sentence.strip().split()[0]

subtask1_dataset = train[['Activity Date','Distance','Moving Time']].copy(deep = True)
subtask1_dataset['Activity Date'] = subtask1_dataset['Activity Date'].apply(monthit)
subtask1_dataset['speed'] = subtask1_dataset['Distance'] / (subtask1_dataset['Moving Time'] / 3600)

def round_special(number):
    dec = Decimal(str(number))
    return dec.quantize(Decimal("0.00001"), rounding=ROUND_DOWN)

months = ['Jan','Feb','Mar','Apr','May','Jun','Jul','Aug','Sep','Oct','Nov','Dec']
subtask1_answers = []
for month in months:
    subtask1_answers.append(float(round_special(np.mean(subtask1_dataset[subtask1_dataset['Activity Date'] == month]['speed'].values))))

SUBTASK 2

In [42]:
from catboost import CatBoostClassifier

train_x = train[['Distance','Elapsed Time','Moving Time','Starting Latitude','Starting Longitude','Finish Latitude','Finish Longitude']].copy(deep = True)
train_x['speed'] = train_x['Distance'] / (train_x['Moving Time'] / 3600)
train_x['Moving Time'] = train_x['Moving Time'] / 3600
train_x['Elapsed Time'] = train_x['Elapsed Time'] / 3600
train_y = train['Label']

train_x_values = train_x.values
train_y_values = train_y.values

model = CatBoostClassifier(n_estimators = 2000, learning_rate = 0.005, l2_leaf_reg= 0.3, rsm = 0.9, max_depth = 6,random_state = 0)
model.fit(train_x_values, train_y_values)

0:	learn: 1.0913445	total: 4.99ms	remaining: 9.97s
1:	learn: 1.0842122	total: 11.8ms	remaining: 11.7s
2:	learn: 1.0777851	total: 14.9ms	remaining: 9.92s
3:	learn: 1.0711975	total: 19.8ms	remaining: 9.89s
4:	learn: 1.0643555	total: 25.5ms	remaining: 10.2s
5:	learn: 1.0576495	total: 28.7ms	remaining: 9.55s
6:	learn: 1.0505714	total: 32.2ms	remaining: 9.16s
7:	learn: 1.0449666	total: 35.2ms	remaining: 8.76s
8:	learn: 1.0393414	total: 38.7ms	remaining: 8.57s
9:	learn: 1.0326615	total: 42.7ms	remaining: 8.51s
10:	learn: 1.0267034	total: 45.9ms	remaining: 8.29s
11:	learn: 1.0199111	total: 49.5ms	remaining: 8.2s
12:	learn: 1.0130074	total: 52.6ms	remaining: 8.04s
13:	learn: 1.0076746	total: 55.6ms	remaining: 7.89s
14:	learn: 1.0014444	total: 59.7ms	remaining: 7.9s
15:	learn: 0.9952574	total: 63.1ms	remaining: 7.83s
16:	learn: 0.9909431	total: 66.4ms	remaining: 7.75s
17:	learn: 0.9869157	total: 69.2ms	remaining: 7.62s
18:	learn: 0.9818761	total: 71.7ms	remaining: 7.47s
19:	learn: 0.9763380	tot

CatBoostClassifier(l2_leaf_reg=0.3, learning_rate=0.005, max_depth=6, n_estimators=2000, random_state=0, rsm=0.9)

In [43]:
test_x = test[['Distance','Elapsed Time','Moving Time','Starting Latitude','Starting Longitude','Finish Latitude','Finish Longitude']].copy(deep = True)
test_x['speed'] = test_x['Distance'] / (test_x['Moving Time'] / 3600)
test_x['Moving Time'] = test_x['Moving Time'] / 3600
test_x['Elapsed Time'] = test_x['Elapsed Time'] / 3600
test_x_values = test_x.values

subtask2_answers = model.predict(test_x_values).flatten()

SUBMISSION

In [44]:
p1 = pd.DataFrame({
    'subtaskID':1,
    'Answer1':months,
    'Answer2':subtask1_answers
})
p2 = pd.DataFrame({
    'subtaskID':2,
    'Answer1':test['Activity ID'],
    'Answer2': subtask2_answers
})

pd.concat([p1,p2]).to_csv('submission.csv',index=False)